# Khảo Sát Metadata & Cấu Trúc Mẫu 5 Bộ Dữ Liệu BCTC (Hugging Face)

**Dự án**: Hệ thống tự động trích xuất Báo cáo tài chính & Đánh giá sức khỏe doanh nghiệp Việt Nam  
**Giai đoạn 1**: Khảo sát Metadata, phân tích định dạng tệp, giấy phép bản quyền và kiểm thử mẫu nhỏ.

### Danh mục 5 bộ dữ liệu khảo sát:
1. `vduydong/ocr_annual_financials` (Văn bản OCR BCTC doanh nghiệp niêm yết theo năm)
2. `conghuy/ocr_financials_statements_2020_2025` (Văn bản OCR BCTC kiểm toán giai đoạn 2020 - 2025)
3. `VietFinTabGroup/VietFinTab` (Cấu trúc bảng biểu & tọa độ Bounding Box BCTC tiếng Việt)
4. `AIGuruTinix/ViFinQA` (Bộ hỏi đáp tài chính & danh mục mã cổ phiếu)
5. `bsmock/FinTabNet.c` (Benchmark quốc tế về cấu trúc bảng phức tạp)

> **Lưu ý an toàn tài nguyên**: Toàn bộ notebook này sử dụng Hugging Face Hub REST API và cơ chế tải mẫu đơn lẻ (1-2 file đại diện dung lượng vài chục KB). **Tuyệt đối không tải toàn bộ dataset** (đặc biệt `FinTabNet.c` nặng >3.4 GB và các tập hàng chục nghìn file).


In [ ]:
# Thiết lập thư viện và cấu hình đường dẫn tương thích Windows
import os
import sys
import json
import urllib.request
from pathlib import Path
import pandas as pd

# Đường dẫn dự án tương thích tuyệt đối trên Windows (sử dụng pathlib.Path)
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / 'data'
SAMPLES_DIR = DATA_DIR / 'samples'

# Đảm bảo thư mục lưu mẫu tồn tại
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Thư mục gốc dự án: {PROJECT_ROOT}")
print(f"Thư mục chứa dữ liệu mẫu: {SAMPLES_DIR}")
print(f"Phiên bản Python hiện tại: {sys.version.split()[0]}")


## 1. Khảo Sát Metadata Toàn Diện Qua Hugging Face API

Chúng ta gọi trực tiếp REST API `https://huggingface.co/api/datasets/{dataset_id}` để lấy:
- Giấy phép bản quyền (License)
- Tổng số lượng file trong kho lưu trữ (Sibling files count)
- Các thẻ tác vụ (Task tags)
- Tình trạng công khai/riêng tư (Private status)
- Lượt tải (Downloads) và lượt yêu thích (Likes)


In [ ]:
# Danh sách 5 dataset cần khảo sát
DATASET_IDS = [
    'vduydong/ocr_annual_financials',
    'conghuy/ocr_financials_statements_2020_2025',
    'VietFinTabGroup/VietFinTab',
    'AIGuruTinix/ViFinQA',
    'bsmock/FinTabNet.c'
]

def fetch_hf_metadata(dataset_id: str) -> dict:
    '''Truy vấn thông tin metadata của một dataset trên Hugging Face API.'''
    api_url = f'https://huggingface.co/api/datasets/{dataset_id}'
    headers = {'User-Agent': 'FinancialHealthSystem/1.0 (Windows)'}
    
    try:
        req = urllib.request.Request(api_url, headers=headers)
        with urllib.request.urlopen(req, timeout=15) as response:
            if response.status == 200:
                data = json.loads(response.read().decode('utf-8'))
                card_data = data.get('cardData') or {}
                siblings = data.get('siblings', [])
                
                # Trích xuất các phần mở rộng file có trong dataset
                extensions = set()
                for s in siblings:
                    fname = s.get('rfilename', '')
                    if '.' in fname:
                        ext = fname.split('.')[-1].lower()
                        extensions.add(ext)
                
                return {
                    'Dataset ID': dataset_id,
                    'License': card_data.get('license', 'Chưa khai báo'),
                    'Total Files': len(siblings),
                    'File Formats': ', '.join(sorted(list(extensions))[:6]),
                    'Private': data.get('private', False),
                    'Downloads': data.get('downloads', 0),
                    'Likes': data.get('likes', 0),
                    'Tags': ', '.join([t for t in data.get('tags', []) if not t.startswith('license:')][:3])
                }
    except Exception as e:
        return {
            'Dataset ID': dataset_id,
            'Error': str(e)
        }

# Thu thập metadata của 5 datasets
print("Đang truy vấn metadata từ Hugging Face Hub API...")
metadata_list = [fetch_hf_metadata(ds) for ds in DATASET_IDS]
df_metadata = pd.DataFrame(metadata_list)
display(df_metadata)


## 2. Khảo Sát Chi Tiết Từng Bộ Dữ Liệu & Tải Mẫu Thử Nghiệm Nhỏ

Dưới đây ta sẽ tải **chính xác 1 tệp mẫu đại diện (vài chục KB)** từ mỗi dataset để khảo sát schema, cấu trúc văn bản hoặc tọa độ bảng biểu. Tệp mẫu được lưu trữ an toàn trong thư mục `data/samples/`.


### 2.1. Dataset 1: `vduydong/ocr_annual_financials`
- **Mục tiêu**: BCTC doanh nghiệp niêm yết theo năm.
- **Cấu trúc lưu trữ**: `{Mã CK}/{Năm}/{Tên báo cáo}/{Tên báo cáo}_extracted.txt`.
- **Thử nghiệm**: Tải 1 file OCR của công ty Cổ phần 32 (Mã A32) năm 2017.


In [ ]:
sample_url_vduydong = 'https://huggingface.co/datasets/vduydong/ocr_annual_financials/resolve/main/A32/2017/A32_Baocaotaichinh_2017_Kiemtoan/A32_Baocaotaichinh_2017_Kiemtoan_extracted.txt'
dest_vduydong = SAMPLES_DIR / 'vduydong_A32_2017_sample.txt'

try:
    print(f"Đang tải tệp mẫu vduydong: {dest_vduydong.name}...")
    headers = {'User-Agent': 'FinancialHealth/1.0'}
    req = urllib.request.Request(sample_url_vduydong, headers=headers)
    with urllib.request.urlopen(req, timeout=15) as res, open(dest_vduydong, 'wb') as f_out:
        f_out.write(res.read())
        
    file_size_kb = dest_vduydong.stat().st_size / 1024
    print(f"-> Tải thành công! Dung lượng: {file_size_kb:.2f} KB")
    
    # Đọc và hiển thị 25 dòng đầu tiên
    with open(dest_vduydong, 'r', encoding='utf-8', errors='ignore') as f:
        sample_lines = [next(f).rstrip() for _ in range(25)]
    
    print("\n--- NỘI DUNG 25 DÒNG ĐẦU TIÊN CỦA VDUYDONG SAMPLE ---")
    print("\n".join(sample_lines))
except Exception as e:
    print(f"Lỗi tải mẫu vduydong: {e}")


### 2.2. Dataset 2: `conghuy/ocr_financials_statements_2020_2025`
- **Mục tiêu**: BCTC kiểm toán giai đoạn gần nhất (2020 - 2025).
- **Cấu trúc lưu trữ**: `ocr_results/{Mã CK}/{Năm}/{Mã CK}_Baocaotaichinh_{Năm}_{Loại}_extracted.txt`.
- **Thử nghiệm**: Tải mẫu BCTC năm 2020 của mã A32 để so sánh với dữ liệu của tác giả vduydong.


In [ ]:
sample_url_conghuy = 'https://huggingface.co/datasets/conghuy/ocr_financials_statements_2020_2025/resolve/main/ocr_results/A32/2020/A32_Baocaotaichinh_2020_Kiemtoan_extracted.txt'
dest_conghuy = SAMPLES_DIR / 'conghuy_A32_2020_sample.txt'

try:
    print(f"Đang tải tệp mẫu conghuy: {dest_conghuy.name}...")
    headers = {'User-Agent': 'FinancialHealth/1.0'}
    req = urllib.request.Request(sample_url_conghuy, headers=headers)
    with urllib.request.urlopen(req, timeout=15) as res, open(dest_conghuy, 'wb') as f_out:
        f_out.write(res.read())
        
    file_size_kb = dest_conghuy.stat().st_size / 1024
    print(f"-> Tải thành công! Dung lượng: {file_size_kb:.2f} KB")
    
    with open(dest_conghuy, 'r', encoding='utf-8', errors='ignore') as f:
        sample_lines = [next(f).rstrip() for _ in range(25)]
        
    print("\n--- NỘI DUNG 25 DÒNG ĐẦU TIÊN CỦA CONGHUY SAMPLE ---")
    print("\n".join(sample_lines))
except Exception as e:
    print(f"Lỗi tải mẫu conghuy: {e}")


### 2.3. Dataset 3: `VietFinTabGroup/VietFinTab`
- **Mục tiêu**: Nhận diện cấu trúc bảng (Table Structure Recognition) cho BCTC tiếng Việt.
- **Cấu trúc lưu trữ**: Mỗi trang tài liệu có 1 file ảnh `.jpg` và 1 file `.json` chú thích tọa độ `bbox` kèm text.
- **Thử nghiệm**: Tải file chú thích JSON mẫu `EVF_CN_2023_21.json` và phân tích các trường nhãn OCR.


In [ ]:
sample_url_vietfintab = 'https://huggingface.co/datasets/VietFinTabGroup/VietFinTab/resolve/main/contents/EVF/EVF_CN_2023_21.json'
dest_vietfintab = SAMPLES_DIR / 'vietfintab_EVF_2023_sample.json'

try:
    print(f"Đang tải tệp mẫu VietFinTab: {dest_vietfintab.name}...")
    headers = {'User-Agent': 'FinancialHealth/1.0'}
    req = urllib.request.Request(sample_url_vietfintab, headers=headers)
    with urllib.request.urlopen(req, timeout=15) as res, open(dest_vietfintab, 'wb') as f_out:
        f_out.write(res.read())
        
    print(f"-> Tải thành công! Dung lượng: {dest_vietfintab.stat().st_size / 1024:.2f} KB")
    
    # Parse cấu trúc JSON
    with open(dest_vietfintab, 'r', encoding='utf-8') as f:
        json_data = json.loads(f.read())
    
    ocr_items = json_data.get('ocr', [])
    print(f"Tổng số mục OCR nhận diện được trong bảng: {len(ocr_items)}")
    
    # Hiển thị 10 mục đầu tiên dưới dạng DataFrame
    df_ocr_sample = pd.DataFrame(ocr_items[:10])
    print("\n--- 10 MỤC OCR ĐẦU TIÊN CỦA VIETFINTAB ---")
    display(df_ocr_sample)
except Exception as e:
    print(f"Lỗi phân tích mẫu VietFinTab: {e}")


### 2.4. Dataset 4: `AIGuruTinix/ViFinQA`
- **Mục tiêu**: Bộ dữ liệu Hỏi - Đáp tài chính (Financial QA) phục vụ đánh giá tính chính xác của dữ liệu trích xuất.
- **Thử nghiệm**:
  1. Tải `code_stock.csv`: Danh sách mã cổ phiếu các công ty.
  2. Đọc trực tiếp 5 bản ghi đầu tiên trong `questions/questions.jsonl` qua stream HTTP không tải cả file.


In [ ]:
# 1. Tải và khảo sát code_stock.csv
url_code_stock = 'https://huggingface.co/datasets/AIGuruTinix/ViFinQA/resolve/main/code_stock.csv'
dest_code_stock = SAMPLES_DIR / 'code_stock.csv'

try:
    headers = {'User-Agent': 'FinancialHealth/1.0'}
    req = urllib.request.Request(url_code_stock, headers=headers)
    with urllib.request.urlopen(req, timeout=15) as res, open(dest_code_stock, 'wb') as f_out:
        f_out.write(res.read())
        
    df_stocks = pd.read_csv(dest_code_stock)
    print(f"Danh sách mã cổ phiếu trong ViFinQA (Tổng cộng: {len(df_stocks)} mã):")
    display(df_stocks.head(10))
except Exception as e:
    print(f"Lỗi tải code_stock.csv: {e}")

# 2. Đọc stream 5 câu hỏi đầu tiên trong questions.jsonl
url_questions = 'https://huggingface.co/datasets/AIGuruTinix/ViFinQA/resolve/main/questions/questions.jsonl'
try:
    req = urllib.request.Request(url_questions, headers={'User-Agent': 'FinancialHealth/1.0'})
    print("\n--- 5 CÂU HỎI TÀI CHÍNH MẪU TRONG VIFINQA (QUESTIONS.JSONL) ---")
    with urllib.request.urlopen(req, timeout=15) as res:
        sample_questions = []
        for _ in range(5):
            line = res.readline().decode('utf-8').strip()
            if line:
                sample_questions.append(json.loads(line))
        display(pd.DataFrame(sample_questions))
except Exception as e:
    print(f"Lỗi đọc questions.jsonl: {e}")


### 2.5. Dataset 5: `bsmock/FinTabNet.c`
- **Mục tiêu**: Benchmark nhận diện cấu trúc bảng quốc tế (Table Structure Recognition).
- **CẢNH BÁO DUNG LƯỢNG**:
  - `FinTabNet.c-Structure.tar.gz`: **~3.17 GB**
  - `FinTabNet.c-PDF_Annotations.tar.gz`: **~255 MB**
- **Quy tắc an toàn**: **Tuyệt đối KHÔNG tải 2 tệp nén lớn này**. Chúng ta đọc trực tiếp `README.md` của repo để khảo sát cách gán nhãn XML/HTML/bbox cấu trúc bảng phục vụ chuẩn hóa mô hình sau này.


In [ ]:
url_fintabnet_readme = 'https://huggingface.co/datasets/bsmock/FinTabNet.c/resolve/main/README.md'

try:
    req = urllib.request.Request(url_fintabnet_readme, headers={'User-Agent': 'FinancialHealth/1.0'})
    with urllib.request.urlopen(req, timeout=15) as res:
        fintabnet_readme = res.read().decode('utf-8')
    
    print("--- ĐẶC TẢ CẤU TRÚC FINTABNET.C (TRÍCH TỪ README) ---")
    print("\n".join(fintabnet_readme.split("\n")[:30]))
except Exception as e:
    print(f"Lỗi đọc README FinTabNet.c: {e}")


## 3. Tổng Kết & Ma Trận Đánh Giá Vai Trò Trong Dự Án

| Tên Dataset | Vai trò cốt lõi trong hệ thống | Ưu điểm chính | Hạn chế / Điểm cần lưu ý |
|---|---|---|---|
| `vduydong/ocr_annual_financials` | Kho văn bản BCTC lịch sử | Số lượng lớn (>18k file), đa dạng ngành nghề | Chất lượng OCR phụ thuộc bản PDF gốc, thiếu phân loại bảng chi tiết |
| `conghuy/ocr_financials_statements_2020_2025` | Kho văn bản BCTC hiện tại (2020-2025) | Cập nhật số liệu mới nhất, có đánh dấu phân trang rõ ràng | Cần module regex/NLP để bóc tách 3 bảng chính |
| `VietFinTabGroup/VietFinTab` | Dữ liệu huấn luyện/kiểm định cấu trúc bảng | Có tọa độ bounding box và text tiếng Việt chuẩn xác | Chủ yếu tập trung vào phát hiện bảng (detection) |
| `AIGuruTinix/ViFinQA` | Bộ câu hỏi kiểm thử tính toàn vẹn (Golden QA) | Câu hỏi thực tế (doanh thu, lợi nhuận, tiền gửi), định dạng JSONL chuẩn | Cần đối chiếu với BCTC tương ứng |
| `bsmock/FinTabNet.c` | Bộ quy chuẩn cấu trúc bảng nâng cao | Chuẩn quốc tế cho cấu trúc bảng phức tạp (cell span, row span) | Dung lượng rất lớn (>3.4 GB), dữ liệu tiếng Anh |

---
### Kế Hoạch Đề Xuất Cho Giai Đoạn 2:
1. Xây dựng **Parser Pipeline** (`src/data/parser.py`) để bóc tách 3 bảng chuẩn từ văn bản OCR:
   - Bảng Cân đối kế toán (Balance Sheet)
   - Báo cáo Kết quả hoạt động kinh doanh (Income Statement)
   - Báo cáo Lưu chuyển tiền tệ (Cash Flow Statement)
2. Chuẩn hóa chỉ tiêu sang chuẩn kế toán Việt Nam (Thông tư 200/2014/TT-BTC).
3. Viết module kiểm thử tính cân đối tài chính (`Tổng Tài sản = Tổng Nguồn vốn`) trước khi đưa vào mô hình đánh giá sức khỏe.
